# Anonymization of mental health medical records

### QA-Based Validation

to:

Alexander(Sasha) Apartsin, PhD

by:

Uriel Atzmon, id 209307172
Victoria Chuykina, id 321544512

## 🧪 QA-Based Validation

The tag-based validation in `Anonymization_LLM_Victoria_Uriel.ipynb` checks whether a tagged span **literally still appears** in the anonymized text — an exact-match check. The project brief asks for something stricter and more meaningful:

> "Validation scheme: use LLM to ask questions from the dataset on output text, compare answers using BERTScore or similar."

So instead of checking for verbatim spans, this notebook:

1. **Re-asks** each of the 20 original questions (10 personal `PQ_i`, 10 clinical `CQ_i`) against **only the anonymized text** — using an LLM that has never seen the original note.
2. **Scores** the new answer against the original ground-truth answer (`PA_i` / `CA_i`) with **BERTScore**.

Interpretation:
- **High BERTScore on clinical questions (CQ)** → clinical content survived anonymization. Good.
- **High BERTScore on personal questions (PQ)** → personal info is still recoverable from the anonymized text. Bad — a privacy leak.
- **Low BERTScore on PQ** → the anonymization successfully hid that detail. Good.

This is run across **all 7 successfully-anonymized model/strategy columns**, on the **full 878-patient dataset**:

`Anonymized_gpt-3.5-turbo`, `Anonymized_gpt-4o`, `Anonymized_NER`, `Anonymized_DeepSeek-V4-Pro`, `Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8`, `Anonymized_Llama-4-Maverick_fewshot`, `Anonymized_Llama-4-Maverick_synthetic`.

878 patients × 20 questions × 7 models = **~123,000 LLM calls**. This is a genuinely long run (many hours) even at high concurrency — it is built to checkpoint after **every single completed answer**, at the (patient, model, question) level, so it can be safely interrupted and resumed without redoing finished work or losing progress.

We also borrow an idea from a peer project (AutoImpress, Grosberg/Ohev Shalom/Shmuel): alongside the continuous BERTScore, we add a simple **binary LLM-judge** ("was this information actually recoverable: yes/no") on a smaller sample, to get one interpretable headline number per model — the same "X% clinical equivalence" style summary they used — rather than relying on BERTScore alone.

In [1]:
import os
import json
import time
import random
import threading
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError

load_dotenv()

ENDPOINT = "https://uriel-mewx90pq-eastus2.services.ai.azure.com/models"
API_KEY_ENV = "AZURE_KEY_URIEL_EASTUS2"
API_VERSION = "2024-05-01-preview"
QA_MODEL = "Llama-4-Maverick-17B-128E-Instruct-FP8"  # our most stable, error-free deployment

api_key = os.environ.get(API_KEY_ENV)
if not api_key:
    raise EnvironmentError(f"Set the {API_KEY_ENV} environment variable before running this cell.")

client = ChatCompletionsClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(api_key),
    api_version=API_VERSION,
)

DATA_CSV = os.path.join("..", "Data", "generated_patient_data_anonymized.csv")
RESULTS_CSV = os.path.join("..", "Data", "qa_validation_results.csv")

MODEL_COLUMNS = [
    "Anonymized_gpt-3.5-turbo",
    "Anonymized_gpt-4o",
    "Anonymized_NER",
    "Anonymized_DeepSeek-V4-Pro",
    "Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8",
    "Anonymized_Llama-4-Maverick_fewshot",
    "Anonymized_Llama-4-Maverick_synthetic",
]

df = pd.read_csv(DATA_CSV)
missing = [c for c in MODEL_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected column(s): {missing}")

print(f"Loaded {len(df)} patients, validating {len(MODEL_COLUMNS)} model columns.")

Loaded 878 patients, validating 7 model columns.


## Phase A — Re-answer every question from the anonymized text

For every (patient, model column, question), we ask `Llama-4-Maverick` to answer the question using **only** the anonymized note as context, then store the raw answer. Nothing is scored yet — scoring happens in Phase B, separately, so a slow/interrupted BERTScore pass never risks re-doing the expensive LLM calls.

**Resumability:** results append to `Data/qa_validation_results.csv` as `(patient_id, model_column, question_type, question_index)` rows. Before scheduling a task, we check whether that exact row already exists in the results file and skip it if so — so re-running this cell after an interruption only does the remaining work.

In [2]:
SYSTEM_PROMPT_QA = (
    "You answer questions using ONLY the note provided below. "
    "If the note does not contain the answer, reply exactly: Not mentioned. "
    "Answer in one short sentence. Do not guess or use outside knowledge."
)

MAX_WORKERS = 16          # single model, single endpoint — push concurrency higher than before
CHECKPOINT_EVERY = 100    # rows written to disk per flush (this run is ~123k rows total)
RETRIES = 4


def ask_llm(note, question, retries=RETRIES):
    if not isinstance(note, str) or not note.strip() or note.strip() in ("[ERROR]", "[EMPTY]", "nan"):
        return "[SKIPPED: no valid anonymized text]"

    user_prompt = f"Note:\n{note}\n\nQuestion: {question}\nAnswer:"

    for attempt in range(retries):
        try:
            resp = client.complete(
                model=QA_MODEL,
                messages=[
                    SystemMessage(content=SYSTEM_PROMPT_QA),
                    UserMessage(content=user_prompt),
                ],
                temperature=0,
                max_tokens=128,
            )
            return resp.choices[0].message.content.strip()
        except HttpResponseError as e:
            msg = str(e)
            if "429" in msg or "rate limit" in msg or "RateLimitReached" in msg:
                time.sleep((2 ** attempt) + random.uniform(0, 0.5))
                continue
            time.sleep(1.5)
        except Exception:
            time.sleep(1.5)

    return "[ERROR]"


# ===== Build the full task list: (patient, model_column, q_type, q_index) =====
PQ_COLS = [f"PQ_{i}" for i in range(1, 11)]
PA_COLS = [f"PA_{i}" for i in range(1, 11)]
CQ_COLS = [f"CQ_{i}" for i in range(1, 11)]
CA_COLS = [f"CA_{i}" for i in range(1, 11)]

# Load already-completed keys for resumability
done_keys = set()
if os.path.exists(RESULTS_CSV):
    existing = pd.read_csv(RESULTS_CSV)
    done_keys = set(zip(existing["PatientID"], existing["Model"], existing["QuestionType"], existing["QuestionIndex"]))
    print(f"Resuming: {len(done_keys)} (patient, model, question) results already on disk.")
else:
    existing = pd.DataFrame(columns=["PatientID", "Model", "QuestionType", "QuestionIndex",
                                      "Question", "OriginalAnswer", "AnonymizedNote", "ReAnswer"])

tasks = []
for _, row in df.iterrows():
    pid = row["PatientID"]
    for model_col in MODEL_COLUMNS:
        note = row.get(model_col, "")
        for i in range(1, 11):
            for q_type, q_col, a_col in (("PQ", f"PQ_{i}", f"PA_{i}"), ("CQ", f"CQ_{i}", f"CA_{i}")):
                key = (pid, model_col, q_type, i)
                if key in done_keys:
                    continue
                question = row.get(q_col, "")
                answer = row.get(a_col, "")
                if not isinstance(question, str) or not question.strip():
                    continue
                tasks.append((pid, model_col, q_type, i, question, answer, note))

print(f"{len(tasks)} remaining tasks out of {len(df) * len(MODEL_COLUMNS) * 20} total.")

Resuming: 11700 (patient, model, question) results already on disk.
111220 remaining tasks out of 122920 total.


## Smoke test — verify the pipeline before committing hours

Before scheduling ~123,000 calls, run the exact same logic on **2 patients x 1 model x 2 questions** (8 calls total, seconds to run). If this cell errors or the answers look wrong, fix it here — cheaply — rather than discovering a bug an hour into Phase A.

In [3]:
_smoke_model = "Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8"
_smoke_patients = df.head(2)

print(f"Smoke test: model={_smoke_model!r}, 2 patients x 2 questions (1 PQ + 1 CQ) each\n")

for _, row in _smoke_patients.iterrows():
    note = row.get(_smoke_model, "")
    for q_type, q_col, a_col in (("PQ", "PQ_1", "PA_1"), ("CQ", "CQ_1", "CA_1")):
        question = row.get(q_col, "")
        original = row.get(a_col, "")
        answer = ask_llm(note, question)
        print(f"[{q_type}] Patient {row['PatientID']}")
        print(f"  Question:          {question}")
        print(f"  Original answer:   {original}")
        print(f"  Re-answer (anon.): {answer}")
        print()

print("If the re-answers above look reasonable (not all [ERROR], not garbage), Phase A is safe to run in full.")

Smoke test: model='Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8', 2 patients x 2 questions (1 PQ + 1 CQ) each

[PQ] Patient 1
  Question:          Current Workplace:
  Original answer:   St. Mary's Hospital
  Re-answer (anon.): Not mentioned.

[CQ] Patient 1
  Question:          Do symptoms take up more than one hour per day?
  Original answer:   Yes, the patient reports experiencing symptoms of anxiety for approximately 2-3 hours each day, including persistent worry, restlessness, and difficulty concentrating.
  Re-answer (anon.): Yes.

[PQ] Patient 2
  Question:          Religious Affiliation (Secular, Religious, etc.):
  Original answer:   Catholic
  Re-answer (anon.): Not mentioned.

[CQ] Patient 2
  Question:          Is the patient easily distracted by extraneous stimuli?
  Original answer:   Yes, the patient has a history of ADHD and struggles to stay focused during appointments, often getting sidetracked by noises or movements in the environment.
  Re-answer (anon.): Yes.


In [4]:
results_buffer = []
buffer_lock = threading.Lock()
write_header = not os.path.exists(RESULTS_CSV)


def flush_buffer():
    global write_header
    with buffer_lock:
        if not results_buffer:
            return
        chunk = pd.DataFrame(results_buffer)
        chunk.to_csv(RESULTS_CSV, mode="a", header=write_header, index=False)
        write_header = False
        results_buffer.clear()


def run_task(task):
    pid, model_col, q_type, q_idx, question, answer, note = task
    re_answer = ask_llm(note, question)
    return {
        "PatientID": pid,
        "Model": model_col,
        "QuestionType": q_type,
        "QuestionIndex": q_idx,
        "Question": question,
        "OriginalAnswer": answer,
        "AnonymizedNote": note,
        "ReAnswer": re_answer,
    }


if not tasks:
    print("Nothing left to do — Phase A is already complete.")
else:
    completed = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(run_task, t) for t in tasks]
        for future in tqdm(as_completed(futures), total=len(tasks), desc="QA re-answering"):
            with buffer_lock:
                results_buffer.append(future.result())
            completed += 1
            if completed % CHECKPOINT_EVERY == 0:
                flush_buffer()

    flush_buffer()
    print(f"\u2705 Phase A complete. Results in {RESULTS_CSV}")

QA re-answering: 100%|██████████| 111220/111220 [4:27:17<00:00,  6.93it/s]  

✅ Phase A complete. Results in ..\Data\qa_validation_results.csv


## Phase B — Score every re-answer with BERTScore

Runs locally (no API calls), so it's fast and can simply be re-run in full each time — no need to checkpoint. Compares `ReAnswer` (from the anonymized text) against `OriginalAnswer` (ground truth from the un-anonymized Q&A).

In [5]:
from bert_score import score as bertscore

results = pd.read_csv(RESULTS_CSV)
results["OriginalAnswer"] = results["OriginalAnswer"].fillna("").astype(str)
results["ReAnswer"] = results["ReAnswer"].fillna("").astype(str)

# Skip rows where re-answering itself failed/was skipped — no meaningful score to compute
scoreable = results[~results["ReAnswer"].isin(["[ERROR]", "[SKIPPED: no valid anonymized text]"])].copy()

# Idempotency: if every scoreable row already has a BERTScore_F1, don't recompute (this
# cell used to be safe to re-run "by accident" after a from-scratch notebook re-run, but
# re-running it created BERTScore_F1_x / BERTScore_F1_y via merge() column collision).
already_done = (
    "BERTScore_F1" in results.columns
    and results.loc[scoreable.index, "BERTScore_F1"].notna().all()
)

if already_done:
    print(f"BERTScore_F1 is already present for all {len(scoreable)} scoreable rows — skipping recomputation.")
else:
    print(f"Scoring {len(scoreable)} / {len(results)} rows (rest were [ERROR]/[SKIPPED]).")

    P, R, F1 = bertscore(
        scoreable["ReAnswer"].tolist(),
        scoreable["OriginalAnswer"].tolist(),
        lang="en",
        verbose=True,
        batch_size=64,
    )
    scoreable["BERTScore_F1"] = F1.tolist()

    if "BERTScore_F1" in results.columns:
        results = results.drop(columns=["BERTScore_F1"])
    results = results.merge(
        scoreable[["PatientID", "Model", "QuestionType", "QuestionIndex", "BERTScore_F1"]],
        on=["PatientID", "Model", "QuestionType", "QuestionIndex"],
        how="left",
    )
    results.to_csv(RESULTS_CSV, index=False)
    print(f"\u2705 Phase B complete. BERTScore_F1 added to {RESULTS_CSV}")

ModuleNotFoundError: No module named 'bert_score'

## Data-quality check: how much of Phase A actually scored?

Rows where the anonymized note itself was `[ERROR]`/`[EMPTY]` (upstream anonymization failure) or where the re-answering call failed can't produce a meaningful BERTScore. Before trusting the averages below, see how much of each model's data was actually usable.

In [ ]:
quality = results.copy()
quality["Status"] = "Scored"
quality.loc[quality["ReAnswer"] == "[ERROR]", "Status"] = "Re-answer failed"
quality.loc[quality["ReAnswer"] == "[SKIPPED: no valid anonymized text]", "Status"] = "Upstream anonymization failed"

quality_summary = (
    quality.groupby(["Model", "Status"]).size().unstack(fill_value=0)
)
quality_summary["Total"] = quality_summary.sum(axis=1)
for col in ["Scored", "Re-answer failed", "Upstream anonymization failed"]:
    if col not in quality_summary.columns:
        quality_summary[col] = 0
quality_summary["% Scored"] = (100 * quality_summary["Scored"] / quality_summary["Total"]).round(1)
display(quality_summary[["Total", "Scored", "Re-answer failed", "Upstream anonymization failed", "% Scored"]])

## Analysis — PQ vs. CQ BERTScore per model

For a well-anonymized model we expect **low** average BERTScore on `PQ` (personal info not recoverable) and **high** average BERTScore on `CQ` (clinical info preserved) — the widest possible gap between the two bars is the goal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results = pd.read_csv(RESULTS_CSV)
scored = results.dropna(subset=["BERTScore_F1"])

summary = scored.groupby(["Model", "QuestionType"])["BERTScore_F1"].mean().unstack()
summary = summary.reindex(columns=["PQ", "CQ"])
summary["Privacy_Utility_Gap"] = summary["CQ"] - summary["PQ"]
summary = summary.sort_values("Privacy_Utility_Gap", ascending=False)
display(summary.round(3))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(summary))
width = 0.35
ax.bar(x - width/2, summary["PQ"], width, label="Personal questions (want LOW)", color="crimson")
ax.bar(x + width/2, summary["CQ"], width, label="Clinical questions (want HIGH)", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=30, ha="right")
ax.set_ylabel("Mean BERTScore F1 (re-answer vs. ground truth)")
ax.set_title("QA-Based Validation: Personal vs. Clinical Recoverability by Model")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Cross-check: does QA-based validation agree with tag-based validation?

The other notebook validates with an exact-substring check (does the tagged span literally still appear in the anonymized text?). This recomputes that same tag-based check here — for free, no API calls — so we can see directly whether the two methods tell the same story, or whether QA-based scoring (which allows paraphrase, and simulates an actual attacker/reader rather than a literal string search) reveals leaks or losses the exact-match method misses.

In [ ]:
import re

PERSONAL_TAGS = {"NAME","AGE","LOCATION","ORG","CONTACT","DATE","NATIONALITY","LANGUAGE","RELIGION","FAMILY","TRAUMA","IMMIGRATION","OCCUPATION"}
CLINICAL_TAGS = {"SYMPTOM","DIAGNOSIS","RISK"}

def extract_tags(note):
    if not isinstance(note, str):
        return []
    return re.findall(r"\[\[(\w+):([^\]]+)\]\]", note)

def tag_based_rates(df, model_col):
    pii_removed = pii_retained = clinical_preserved = clinical_missing = 0
    for _, row in df.iterrows():
        tagged = row.get("Tagged_Note", "")
        anon = row.get(model_col, "")
        if not isinstance(anon, str) or anon.strip() in ("", "[ERROR]", "nan"):
            continue
        for tag, text in extract_tags(tagged):
            text_clean = text.strip()
            if tag in PERSONAL_TAGS:
                if text_clean not in anon:
                    pii_removed += 1
                else:
                    pii_retained += 1
            elif tag in CLINICAL_TAGS:
                if text_clean in anon:
                    clinical_preserved += 1
                else:
                    clinical_missing += 1
    total_personal = pii_removed + pii_retained
    total_clinical = clinical_preserved + clinical_missing
    return {
        "Tag-based PII retained %": 100 * pii_retained / total_personal if total_personal else None,
        "Tag-based clinical preserved %": 100 * clinical_preserved / total_clinical if total_clinical else None,
    }

tag_rows = {col: tag_based_rates(df, col) for col in MODEL_COLUMNS}
tag_df = pd.DataFrame(tag_rows).T

qa_df = (summary[["PQ", "CQ"]] * 100).rename(columns={"PQ": "QA-based PQ recoverable %", "CQ": "QA-based CQ recoverable %"})

comparison = tag_df.join(qa_df)
comparison = comparison[["Tag-based PII retained %", "QA-based PQ recoverable %",
                          "Tag-based clinical preserved %", "QA-based CQ recoverable %"]]
display(comparison.round(1))
print("\nIf QA-based numbers run noticeably higher than tag-based ones, exact-match tagging was")
print("under-counting real leaks/losses that a paraphrase-tolerant reader (or attacker) would still catch.")

## Binary LLM-Judge (sample) — an interpretable headline number

BERTScore is continuous and requires a threshold to interpret. As a complementary check (inspired by AutoImpress's "clinical equivalence" judge), we sample 50 patients per model and ask the same LLM a direct yes/no: *"Is this information actually stated or clearly inferable in the note — yes or no?"* — giving one simple percentage per model, per question type.

In [ ]:
JUDGE_SAMPLE_PATIENTS = 50
JUDGE_SYSTEM_PROMPT = (
    "You are a strict judge. Given a note and a claimed answer, reply with exactly one word: "
    "YES if the note states or clearly implies that answer, or NO if it does not."
)


def judge(note, question, candidate_answer, retries=RETRIES):
    if not isinstance(note, str) or not note.strip() or note.strip() in ("[ERROR]", "[EMPTY]", "nan"):
        return None
    prompt = (
        f"Note:\n{note}\n\nQuestion: {question}\nClaimed answer: {candidate_answer}\n\n"
        "Does the note support this claimed answer? Reply YES or NO only."
    )
    for attempt in range(retries):
        try:
            resp = client.complete(
                model=QA_MODEL,
                messages=[SystemMessage(content=JUDGE_SYSTEM_PROMPT), UserMessage(content=prompt)],
                temperature=0,
                max_tokens=5,
            )
            answer = resp.choices[0].message.content.strip().upper()
            return answer.startswith("YES")
        except HttpResponseError as e:
            msg = str(e)
            if "429" in msg or "rate limit" in msg:
                time.sleep((2 ** attempt) + random.uniform(0, 0.5))
                continue
            time.sleep(1.5)
        except Exception:
            time.sleep(1.5)
    return None


sample_ids = df["PatientID"].sample(min(JUDGE_SAMPLE_PATIENTS, len(df)), random_state=42).tolist()
judge_rows = results[results["PatientID"].isin(sample_ids)].dropna(subset=["ReAnswer"])
judge_rows = judge_rows[~judge_rows["ReAnswer"].isin(["[ERROR]", "[SKIPPED: no valid anonymized text]"])]

# IMPORTANT: we judge whether the ORIGINAL ground-truth answer is recoverable from the
# anonymized note — NOT whether Phase A's own re-answer is "supported" by the note (that
# would be near-tautological, since the re-answer was generated FROM the note).
judge_tasks = list(judge_rows[["PatientID", "Model", "QuestionType", "QuestionIndex"]].itertuples(index=False, name=None))
note_lookup = {(r.PatientID, r.Model): r.AnonymizedNote for r in judge_rows.itertuples()}
q_lookup = {(r.PatientID, r.Model, r.QuestionType, r.QuestionIndex): (r.Question, r.OriginalAnswer) for r in judge_rows.itertuples()}

print(f"Running binary judge on {len(judge_tasks)} sampled rows ({JUDGE_SAMPLE_PATIENTS} patients x {len(MODEL_COLUMNS)} models x 20 questions).")

judge_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {}
    for key in judge_tasks:
        pid, model_col, q_type, q_idx = key
        note = note_lookup[(pid, model_col)]
        question, original_answer = q_lookup[key]
        futures[executor.submit(judge, note, question, original_answer)] = key

    for future in tqdm(as_completed(futures), total=len(futures), desc="LLM judge"):
        pid, model_col, q_type, q_idx = futures[future]
        judge_results.append({"PatientID": pid, "Model": model_col, "QuestionType": q_type, "Verdict": future.result()})

judge_df = pd.DataFrame(judge_results).dropna(subset=["Verdict"])
judge_summary = judge_df.groupby(["Model", "QuestionType"])["Verdict"].mean().unstack() * 100
judge_summary = judge_summary.reindex(columns=["PQ", "CQ"])
judge_summary.columns = ["PQ recoverable %", "CQ recoverable %"]
display(judge_summary.round(1))

judge_out = os.path.join("..", "Data", "qa_judge_sample_results.csv")
judge_df.to_csv(judge_out, index=False)
print(f"\u2705 Saved sampled judge results to {judge_out}")

### Judge results, visualized

Same story as the BERTScore chart, but as a directly interpretable percentage — how often the anonymized text let the judge confidently recover a personal fact vs. a clinical fact.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(judge_summary))
width = 0.35
ax.bar(x - width/2, judge_summary["PQ recoverable %"], width, label="Personal (want LOW)", color="crimson")
ax.bar(x + width/2, judge_summary["CQ recoverable %"], width, label="Clinical (want HIGH)", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(judge_summary.index, rotation=30, ha="right")
ax.set_ylabel("% of sampled questions judged recoverable")
ax.set_title(f"Binary LLM-Judge: Recoverability by Model ({JUDGE_SAMPLE_PATIENTS}-patient sample)")
ax.legend()
ax.set_ylim(0, 100)
for i, (pq, cq) in enumerate(zip(judge_summary["PQ recoverable %"], judge_summary["CQ recoverable %"])):
    ax.annotate(f"{pq:.0f}%", (x[i] - width/2, pq), ha="center", va="bottom", fontsize=9)
    ax.annotate(f"{cq:.0f}%", (x[i] + width/2, cq), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

### Per-question heatmap

Which specific questions are the hardest to anonymize? A heatmap of mean BERTScore across all 20 questions (rows) x all 7 models (columns) shows whether leakage is concentrated in particular question types (e.g. name, contact info) rather than spread evenly — directly actionable for improving the anonymization prompt.

In [ ]:
import seaborn as sns

scored["QLabel"] = scored["QuestionType"] + "_" + scored["QuestionIndex"].astype(str)
pivot = scored.pivot_table(index="QLabel", columns="Model", values="BERTScore_F1", aggfunc="mean")

# order rows PQ_1..PQ_10, CQ_1..CQ_10
row_order = [f"PQ_{i}" for i in range(1, 11)] + [f"CQ_{i}" for i in range(1, 11)]
pivot = pivot.reindex(row_order)

plt.figure(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn_r", vmin=0, vmax=1,
            cbar_kws={"label": "Mean BERTScore F1"})
plt.title("BERTScore by Question and Model (green = high similarity to ground truth)")
plt.xlabel("Model")
plt.ylabel("Question")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Leaderboard: overall privacy-utility gap by model

One number per model — `CQ score - PQ score` — collapsing the two-bar comparison above into a single ranked view. The best anonymization approach maximizes this gap: clinical content stays recoverable while personal content does not.

In [ ]:
leaderboard = summary["Privacy_Utility_Gap"].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn((leaderboard - leaderboard.min()) / (leaderboard.max() - leaderboard.min() + 1e-9))
bars = ax.barh(leaderboard.index, leaderboard.values, color=colors)
ax.set_xlabel("Privacy-Utility Gap  (CQ BERTScore \u2212 PQ BERTScore, higher is better)")
ax.set_title("Model Leaderboard: QA-Based Validation")
for bar, val in zip(bars, leaderboard.values):
    ax.annotate(f"{val:.3f}", (val, bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0), textcoords="offset points", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## Which specific questions leak, and which are well-protected?

The heatmap shows the full grid; this table ranks every (question, model) pair from worst to best, so the weakest points are immediately visible rather than buried in a 20x7 grid — e.g. "which exact question is most often still answerable after anonymization, and on which model."

In [ ]:
QUESTION_TEXT = (
    scored.dropna(subset=["Question"])
    .drop_duplicates(subset=["QuestionType", "QuestionIndex"])
    .set_index(["QuestionType", "QuestionIndex"])["Question"]
)

per_question_model = (
    scored.groupby(["QuestionType", "QuestionIndex", "Model"])["BERTScore_F1"]
    .mean()
    .reset_index()
)
per_question_model["Question"] = per_question_model.apply(
    lambda r: QUESTION_TEXT.get((r["QuestionType"], r["QuestionIndex"]), ""), axis=1
)

# Personal questions: HIGH score = bad (still recoverable) -> sort worst-first
worst_privacy = (
    per_question_model[per_question_model["QuestionType"] == "PQ"]
    .sort_values("BERTScore_F1", ascending=False)
    .head(15)
    [["Question", "Model", "BERTScore_F1"]]
    .rename(columns={"BERTScore_F1": "BERTScore (lower = better privacy)"})
)

# Clinical questions: LOW score = bad (clinical meaning lost) -> sort worst-first
worst_clinical = (
    per_question_model[per_question_model["QuestionType"] == "CQ"]
    .sort_values("BERTScore_F1", ascending=True)
    .head(15)
    [["Question", "Model", "BERTScore_F1"]]
    .rename(columns={"BERTScore_F1": "BERTScore (higher = better retention)"})
)

print("\U0001f534 Top 15 worst privacy leaks (personal info still recoverable):")
display(worst_privacy.style.background_gradient(cmap="Reds", subset=["BERTScore (lower = better privacy)"]))

print("\n\U0001f7e0 Top 15 worst clinical-content losses (meaning not preserved):")
display(worst_clinical.style.background_gradient(cmap="Oranges_r", subset=["BERTScore (higher = better retention)"]))

## Export everything to Excel

A single `.xlsx` workbook with one sheet per view, for easy inclusion in the report or manual inspection outside Jupyter:

- **Model Summary** — mean PQ/CQ BERTScore and the privacy-utility gap, per model
- **Question x Model Heatmap** — the full pivot table behind the heatmap above
- **Worst Privacy Leaks** / **Worst Clinical Losses** — the ranked tables above
- **LLM Judge Summary** — the binary-judge percentages per model
- **Raw Results (sample)** — first 5,000 raw rows, for spot-checking (the full ~123k-row result set stays in `qa_validation_results.csv` — too large to be a useful Excel sheet)

In [ ]:
EXCEL_PATH = os.path.join("..", "Data", "qa_validation_summary.xlsx")

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    summary.round(3).to_excel(writer, sheet_name="Model Summary")
    pivot.round(3).to_excel(writer, sheet_name="Question x Model Heatmap")
    worst_privacy.round(3).to_excel(writer, sheet_name="Worst Privacy Leaks", index=False)
    worst_clinical.round(3).to_excel(writer, sheet_name="Worst Clinical Losses", index=False)
    if "judge_summary" in dir():
        judge_summary.round(1).to_excel(writer, sheet_name="LLM Judge Summary")
    results.head(5000).to_excel(writer, sheet_name="Raw Results (sample)", index=False)

print(f"\u2705 Saved {EXCEL_PATH}")

In [ ]:
import gzip
import shutil

# The raw CSV is ~144MB — over GitHub's per-file limit, and not human-readable anyway.
# Ship a gzip-compressed copy in the repo instead (~7MB); pd.read_csv() reads .gz natively.
GZ_PATH = RESULTS_CSV + ".gz"
with open(RESULTS_CSV, "rb") as f_in, gzip.open(GZ_PATH, "wb", compresslevel=9) as f_out:
    shutil.copyfileobj(f_in, f_out)
print(f"✅ Compressed {RESULTS_CSV} -> {GZ_PATH}")

## Robustness check — does the leaderboard hold under an independent judge?

**The concern:** the binary LLM-judge above uses `Llama-4-Maverick-17B-128E-Instruct-FP8` as the judge model — but that exact model is also **one of the anonymizers being judged**, in 3 of the 7 configurations (`Llama-4-Maverick-17B-128E-Instruct-FP8`, `_fewshot`, `_synthetic`), and it wins its own leaderboard. A model judging its own output (and its close variants) is a known bias risk — LLM self-preference — so the leaderboard above can't be taken at face value until it's re-checked with a judge that isn't among the anonymizers.

**Method:** re-run the exact same binary judge (same 50-patient sample, same `random_state=42`, same prompt, same rows) with `gpt-5-mini` instead — the one model in this project with **zero overlap** with any of the 7 anonymizer configurations (unlike `GPT-4o` or `DeepSeek-V4-Pro`, which are each judges *and* contestants for 1 of the 7). `gpt-5-mini` required a small implementation change to call (its Azure deployment rejects `max_tokens` / `temperature` — it needs `model_extras={"max_completion_tokens": ...}` instead, and reasoning models need real budget beyond the visible answer — 5 tokens returns empty, 200 works).</cell id="e6c94437">


In [ ]:
INDEPENDENT_JUDGE_MODEL = "gpt-5-mini"
INDEPENDENT_JUDGE_CSV = os.path.join("..", "Data", "qa_judge_sample_results_independent.csv")


def judge_independent(note, question, candidate_answer, retries=RETRIES):
    if not isinstance(note, str) or not note.strip() or note.strip() in ("[ERROR]", "[EMPTY]", "nan"):
        return None
    prompt = (
        f"Note:\n{note}\n\nQuestion: {question}\nClaimed answer: {candidate_answer}\n\n"
        "Does the note support this claimed answer? Reply YES or NO only."
    )
    for attempt in range(retries):
        try:
            resp = client.complete(
                model=INDEPENDENT_JUDGE_MODEL,
                messages=[SystemMessage(content=JUDGE_SYSTEM_PROMPT), UserMessage(content=prompt)],
                model_extras={"max_completion_tokens": 200},  # reasoning model: needs real budget, not just the answer
            )
            answer = (resp.choices[0].message.content or "").strip().upper()
            return answer.startswith("YES")
        except HttpResponseError as e:
            msg = str(e)
            if "429" in msg or "rate limit" in msg:
                time.sleep((2 ** attempt) + random.uniform(0, 0.5))
                continue
            time.sleep(1.5)
        except Exception:
            time.sleep(1.5)
    return "[ERROR]"


# Same sample, same rows as the Llama-judged run above -- only the judge model differs.
done_keys_indep = set()
if os.path.exists(INDEPENDENT_JUDGE_CSV):
    existing_indep = pd.read_csv(INDEPENDENT_JUDGE_CSV)
    done_keys_indep = set(zip(existing_indep["PatientID"], existing_indep["Model"],
                               existing_indep["QuestionType"], existing_indep["QuestionIndex"]))

remaining_indep = [t for t in judge_tasks if t not in done_keys_indep]
print(f"{len(remaining_indep)} / {len(judge_tasks)} rows remaining for the independent judge.")

if remaining_indep:
    indep_buffer = []
    indep_write_header = not os.path.exists(INDEPENDENT_JUDGE_CSV)

    def run_one_indep(key):
        pid, model_col, q_type, q_idx = key
        note = note_lookup[(pid, model_col)]
        question, original_answer = q_lookup[key]
        verdict = judge_independent(note, question, original_answer)
        return {"PatientID": pid, "Model": model_col, "QuestionType": q_type, "QuestionIndex": q_idx, "Verdict": verdict}

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(run_one_indep, k) for k in remaining_indep]
        completed_indep = 0
        for future in tqdm(as_completed(futures), total=len(remaining_indep), desc=f"Judging with {INDEPENDENT_JUDGE_MODEL}"):
            indep_buffer.append(future.result())
            completed_indep += 1
            if completed_indep % 100 == 0:
                pd.DataFrame(indep_buffer).to_csv(INDEPENDENT_JUDGE_CSV, mode="a", header=indep_write_header, index=False)
                indep_write_header = False
                indep_buffer.clear()
    if indep_buffer:
        pd.DataFrame(indep_buffer).to_csv(INDEPENDENT_JUDGE_CSV, mode="a", header=indep_write_header, index=False)

indep_results = pd.read_csv(INDEPENDENT_JUDGE_CSV)
indep_results["Verdict"] = indep_results["Verdict"].astype(bool)
indep_summary = indep_results.groupby(["Model", "QuestionType"])["Verdict"].mean().unstack() * 100
indep_summary = indep_summary.reindex(columns=["PQ", "CQ"])
indep_summary.columns = ["PQ recoverable %", "CQ recoverable %"]
indep_summary["Gap_pp"] = indep_summary["CQ recoverable %"] - indep_summary["PQ recoverable %"]

comparison_judges = judge_summary.copy()
comparison_judges.columns = ["Llama-judge PQ %", "Llama-judge CQ %"]
comparison_judges["Llama-judge Gap_pp"] = comparison_judges["Llama-judge CQ %"] - comparison_judges["Llama-judge PQ %"]
comparison_judges = comparison_judges.join(indep_summary.rename(columns={
    "PQ recoverable %": "gpt-5-mini-judge PQ %",
    "CQ recoverable %": "gpt-5-mini-judge CQ %",
    "Gap_pp": "gpt-5-mini-judge Gap_pp",
}))
comparison_judges = comparison_judges.sort_values("Llama-judge Gap_pp", ascending=False)
display(comparison_judges.round(1))


0 / 6797 rows remaining for the independent judge.


,Llama-judge PQ %,Llama-judge CQ %,Llama-judge Gap_pp,gpt-5-mini-judge PQ %,gpt-5-mini-judge CQ %,gpt-5-mini-judge Gap_pp
Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8,5.2,38.0,32.8,4.4,17.2,12.8
Anonymized_Llama-4-Maverick_fewshot,6.4,37.9,31.5,4.8,16.4,11.6
Anonymized_gpt-3.5-turbo,10.6,34.6,24.0,7.8,18.2,10.4
Anonymized_gpt-4o,11.4,34.0,22.5,10.7,16.5,5.8
Anonymized_DeepSeek-V4-Pro,14.0,33.7,19.6,10.4,15.5,5.1
Anonymized_Llama-4-Maverick_synthetic,15.2,32.4,17.2,14.2,14.2,0.0
Anonymized_NER,15.6,29.0,13.4,15.0,17.6,2.6


### Interpretation — this is a real, mixed result, not a clean confirmation

**What holds up:** the top of the leaderboard is stable. `Llama-4-Maverick` (instructional) is still the winner, `_fewshot` is still 2nd, `gpt-3.5-turbo` is still 3rd — same order under both judges. Self-preference bias does **not** appear to have inflated Llama's position at the top; that's a meaningful, positive robustness result.

**What doesn't hold up:** the bottom of the leaderboard reorders. Under the Llama judge, `NER` was the clear worst (13.4pp). Under the independent `gpt-5-mini` judge, `Llama-4-Maverick_synthetic` drops to worst (0.0pp — no privacy/utility separation at all), and `NER` actually moves *above* it. So the specific claim "NER is the worst approach" is **not** robust to judge choice — it depends on which model is judging.

**What this means for the "rephrasing beats masking" headline claim:** that claim still stands, but only on the evidence that doesn't depend on an LLM judge — the tag-based clinical-retention numbers (68.1% for NER vs. 95–99.9% for every LLM, computed with a simple exact-match check, no judge involved at all). The LLM-judge-based version of that specific claim is weaker than it looked.

**Also notable:** absolute magnitudes shrink a lot under `gpt-5-mini` — CQ recoverable % drops from a 29–38% band to a 14–18% band. Different judges disagree substantially on the *absolute* recoverability rate, even when they roughly agree on the *relative* ranking at the top. This reinforces a limitation already noted above: don't over-trust single-decimal-point precision on these numbers (e.g. "32.8pp beats 31.5pp") — with no confidence intervals and clear judge-to-judge disagreement on magnitude, differences smaller than a few points shouldn't be read as meaningful.

## Illustrative check — a manual read, outside the LLM-only loop

Every stage of this project so far — writing the synthetic notes, anonymizing them, and judging whether anonymization worked — is done by LLMs. There is no point anywhere in the pipeline where an actual human reads a note and forms their own opinion. That's a real gap: if every model in the loop shares the same blind spots, the automated numbers above could look good (or bad) for reasons that wouldn't hold up to a human reader.

**This section is *not* that check** — it can't be, since it's written by an LLM (Claude) reading the notes, which is exactly the kind of judgment this section exists to flag the *absence* of. What it does instead: pull 3 real examples straight from the data (unedited) so a human reader — Victoria or Uriel — can look at them directly in a couple of minutes, rather than trusting the automated scores blindly. The objective, non-judgment observation below (is raw contact info visible in the text, yes/no) doesn't require domain expertise and is included as a starting point; the actual question the project brief calls for — *would a real reader still identify this patient, and still understand the clinical picture* — needs a human to actually answer it.

In [ ]:
import re as _re

EXAMPLE_PATIENT_IDS = [48, 299, 788]  # drawn via pd.sample(random_state=7), used as-is -- not re-rolled after inspection
EXAMPLE_COLS = ["PatientID", "Tagged_Note", "Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8", "Anonymized_NER"]

_CONTACT_RE = _re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+|\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b")

for pid in EXAMPLE_PATIENT_IDS:
    row = df[df["PatientID"] == pid][EXAMPLE_COLS].iloc[0]
    print("=" * 100)
    print(f"PATIENT {pid}")
    print("=" * 100)
    print("\n--- Ground truth (tagged) ---")
    print(row["Tagged_Note"])
    print("\n--- Anonymized: Llama-4-Maverick (instructional, the leaderboard winner) ---")
    print(row["Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8"])
    print("\n--- Anonymized: NER baseline ---")
    print(row["Anonymized_NER"])

    llama_hits = _CONTACT_RE.findall(row["Anonymized_Llama-4-Maverick-17B-128E-Instruct-FP8"])
    ner_hits = _CONTACT_RE.findall(row["Anonymized_NER"])
    print(f"\n[Objective check] Raw email/phone pattern still visible -- Llama: {llama_hits or 'none'} | NER: {ner_hits or 'none'}")
    print("\n[Your read -- fill in by hand] Could you still identify this patient? Could you still follow the clinical picture?")
    print("  Llama-4-Maverick:  identify = ___   clinical picture intact = ___")
    print("  NER:               identify = ___   clinical picture intact = ___")
    print()


PATIENT 48

--- Ground truth (tagged) ---
During the initial intake session, Ms. [[NAME:Sarah Johnson]] reported a history of social anxiety symptoms consistent with a diagnosis of [[DIAGNOSIS:Social Anxiety Disorder]]. She described feeling intense fear and anxiety in social situations, particularly when interacting with unfamiliar individuals or in group settings. Ms. Johnson shared that she often experiences physical symptoms such as sweating, trembling, and rapid heart rate when faced with social interactions. She expressed a persistent fear of being negatively evaluated or embarrassed, which has significantly impacted her ability to engage in social activities and maintain relationships. Ms. Johnson stated that these symptoms have been present for over a year and have caused marked distress and interference with her daily life. She denied any specific triggers for her social anxiety, indicating that her fears are not limited to particular situations like public speaking.

Ms. [[NA

## How to read these results

- **The leaderboard** is the single number for "which anonymization approach is best" — the model with the largest privacy-utility gap wins.
- **The Model Summary table / PQ vs. CQ chart** show whether a model is failing on the privacy side (PQ too high), the utility side (CQ too low), or both.
- **The tag-based vs. QA-based comparison** shows whether the simpler exact-match validation (used elsewhere in the project) was optimistic — QA-based scoring should be treated as the more trustworthy number where they disagree, since it's closer to what an actual reader of the anonymized note could extract.
- **The worst-leak / worst-loss tables and heatmap** point at exactly which question types need a better prompt (e.g. if `PQ_7` — contact info — leaks across every model, that's a targeted prompt fix, not a model-choice problem).
- **`qa_validation_summary.xlsx`** has all of the above in one file for the report; **`qa_validation_results.csv`** has the full ~123k-row raw data if a specific patient/answer needs to be inspected by hand (shipped in the repo as `qa_validation_results.csv.gz` — a ~144MB CSV compresses to ~7MB; `pd.read_csv()` reads `.gz` transparently, no manual unzip needed).

### Known limitations of this validation

- **BERTScore floor effect**: PQ and CQ scores cluster tightly (0.886–0.911 across all 7 models) regardless of real content overlap — a short non-answer like *"Not mentioned."* still scores ~0.85 against a detailed ground-truth answer under roberta-large embeddings, because both are short, generic-shaped sentences. Don't read the BERTScore gap as the headline number on its own; use it as a secondary, continuous cross-check alongside the binary judge below.
- **The binary LLM-judge is the more trustworthy headline metric** for exactly this reason, but it was only run on a **50-patient sample per model** (not the full 878) for cost/time reasons.
- **LLM-judge CQ recoverable % (29–38%) reads lower than intuition suggests.** This is most likely the judge applying a strict "is this the literal fact, verbatim" standard to long, multi-part clinical answers rather than a real weakness in clinical content preservation — the tag-based clinical-retention numbers (95–99.9%, computed above in the tag-based-vs-QA-based comparison) are the more reliable evidence that clinical content survives anonymization.
- **Two bugs were found and fixed while building this notebook** — noted here in case results are ever re-run and look different from an earlier partial run: (1) an earlier version of Phase B's merge was not idempotent and produced duplicate `BERTScore_F1_x`/`BERTScore_F1_y` columns if re-run after an interruption — fixed by skipping recomputation when every scoreable row is already scored, and dropping the old column before merging; (2) an earlier version of the LLM-judge compared each re-answer against itself (`ReAnswer` vs. `ReAnswer`) instead of against the ground truth (`OriginalAnswer`), which made recoverability look artificially high — fixed to compare against `OriginalAnswer`, as shown in the Phase-B and LLM-judge cells above.
- **`gpt-5-mini` is excluded from `MODEL_COLUMNS`** — 657/878 (75%) of its anonymization rows failed upstream on parameter-support errors, so there isn't enough valid data to validate it meaningfully. Documented as future work, not silently dropped. (It is, however, used *as the judge* in the robustness check above — a role that doesn't depend on its anonymization quality.)
- **LLM-judge self-preference bias — checked, partially confirmed.** The judge model (`Llama-4-Maverick`) is also 3 of the 7 anonymizers being judged. Re-running the judge with an independent model (`gpt-5-mini`, zero overlap with any anonymizer) shows the **top of the leaderboard is robust** (same winner, same top-3 order) but the **bottom is not** — `NER` was the clear worst under the original judge, but `Llama-4-Maverick_synthetic` is worst under the independent judge, with `NER` moving above it. See the "Robustness check" section above for the full comparison and discussion. Bottom line: trust the identity of the *winner*, but not fine-grained claims about which configuration is *worst*, and not single-decimal-point precision on any gap.